| Component  | Model                  | Job                         |
| ---------- | ---------------------- | --------------------------- |
| Embedding  | `gemini-embedding-001` | Text → vectors              |
| Generation | `gemini-2.5-flash`     | Context + question → answer |


In [1]:

from google import genai

import os

from dotenv import load_dotenv

In [2]:
load_dotenv()

client = genai.Client(
    api_key=os.getenv("GOOGLE_API_KEY")
)

In [3]:

import faiss

import numpy as np

In [4]:
#  LOAD DOCUMENT

In [5]:

file_path = "usa.txt"

with open(file_path, "r", encoding="utf-8") as f:
    document = f.read()

print("Document loaded successfully!")

print("Total characters:", len(document))

Document loaded successfully!
Total characters: 6949


#  CHUNKING

In [7]:

chunk_size = 500
chunk_overlap = 50

chunks = []

start = 0

while start < len(document):

    end = start + chunk_size

    chunk = document[start:end]

    chunks.append(chunk)

    start = end - chunk_overlap


print("\nTotal chunks:", len(chunks))



Total chunks: 16


In [8]:
#  DISPLAY CHUNKS

print("\n================ CHUNKS ================")

for i, chunk in enumerate(chunks):

    print(f"\n--- Chunk {i} ---")
    print(chunk)



================ CHUNKS ================

--- Chunk 0 ---
US Overview of the U.S. Economy
The United States of America possesses the largest economy in the world in terms of nominal GDP, making it the most powerful economic force globally. It operates under a capitalist mixed economy, where the private sector dominates, but the government plays a significant regulatory and fiscal role. With a population of over 335 million people and a high level of technological advancement, the U.S. economy thrives on a foundation of consumer spending, innovation, gl

--- Chunk 1 ---
 a foundation of consumer spending, innovation, global trade, and financial services. It has a highly diversified structure with strong sectors in technology, healthcare, finance, real estate, defense, and agriculture.

U.S. GDP – Size, Composition, and Global Share
As of 2024, the United States’ nominal GDP is estimated to be around $28 trillion USD, accounting for approximately 25% of the global economy. It ranks #1 i


# CREATE GEMINI EMBEDDINGS


In [ ]:
Chunk
   ↓
Embedding Model
   ↓
[0.12, -0.45, 0.77, ...]

In [9]:

result = client.models.embed_content( model="gemini-embedding-001", contents=chunks )

chunk_embeddings = np.array(
    [embedding.values for embedding in result.embeddings],
    dtype="float32"
)

print("\nEmbedding shape:", chunk_embeddings.shape)


Embedding shape: (16, 3072)


In [ ]:
16   → number of chunks
3072 → dimensions of each embedding


#  CREATE FAISS INDEX


### Why IndexFlatL2 is good for your student RAG project

For a simple RAG pipeline, it is a very good choice because it is:

    * Easy to understand
    * Easy to implement
    * Exact search
   

### L2 means Euclidean distance.

In [10]:

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(chunk_embeddings)

print("Vectors stored in FAISS:", index.ntotal)

Vectors stored in FAISS: 16


#  RAG FUNCTION


In [11]:

def answer_question(question):

    # Create embedding for question

    result = client.models.embed_content( model="gemini-embedding-001", contents=[question] )

    question_embedding = np.array( [result.embeddings[0].values], dtype="float32" )

    # FAISS similarity search

    top_k = 3

    distances, indices = index.search( question_embedding, k=top_k)

    # Retrieve relevant chunks

    retrieved_chunks = []

    for i in indices[0]:

        retrieved_chunks.append(chunks[i])


    # Display retrieved chunks

    print("\n================ RETRIEVED CHUNKS ================")

    for i, chunk in enumerate(retrieved_chunks, 1):

        print(f"\n--- Retrieved Chunk {i} ---")
        print(chunk)


    # Create context

    context = "\n\n".join(retrieved_chunks)


    # Create RAG prompt

    prompt = f"""
You are a helpful assistant.

Answer the question ONLY using the context provided below.

If the answer is not available in the context,
say:

"I don't know based on the provided document."

CONTEXT:
{context}

QUESTION:
{question}

Give a concise and accurate answer.
"""

    # Generate answer using Gemini

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )


    return response.text


In [12]:
#  ASK QUESTION

question = "What are the major challenges facing the US economy?"

answer = answer_question(question)



================ RETRIEVED CHUNKS ================

--- Retrieved Chunk 1 ---
widened, with a small percentage of Americans owning a disproportionate share of wealth. Access to affordable healthcare, housing shortages, and a declining labor force participation in some sectors are additional structural issues.

Another growing concern is political polarization, which often stalls crucial economic reforms and budget agreements, as seen in recurrent debt ceiling crises. The U.S. also faces challenges from global competition, especially from China in areas like AI, semiconduc

--- Retrieved Chunk 2 ---
lity, political gridlock, and competition from emerging powers demand careful policy navigation. If successfully addressed, the U.S. will continue to dominate the global economy well into the future.

--- Retrieved Chunk 3 ---
ills. If the U.S. can handle these challenges, it is well-positioned to remain a global economic powerhouse through 2030 and beyond.

Final Summary
The U.S. economy r

#  FINAL ANSWER


In [13]:

print("\n================ FINAL ANSWER ================")

print(answer)


================ FINAL ANSWER ================
The major challenges facing the US economy include rising debt, inequality (with a small percentage of Americans owning a disproportionate share of wealth), political polarization/gridlock, and global competition (especially from China in areas like AI and semiconductors, and from emerging powers). Additional structural issues include access to affordable healthcare, housing shortages, and declining labor force participation in some sectors.


In [14]:
question = "What is the GDP of USA in 2024?"

answer = answer_question(question)



================ RETRIEVED CHUNKS ================

--- Retrieved Chunk 1 ---
 a foundation of consumer spending, innovation, global trade, and financial services. It has a highly diversified structure with strong sectors in technology, healthcare, finance, real estate, defense, and agriculture.

U.S. GDP – Size, Composition, and Global Share
As of 2024, the United States’ nominal GDP is estimated to be around $28 trillion USD, accounting for approximately 25% of the global economy. It ranks #1 in the world by nominal GDP, far ahead of China (which ranks 2nd). The U.S. GD

--- Retrieved Chunk 2 ---
he U.S. economy has enjoyed consistent long-term growth, averaging around 2-3% annually. Post-pandemic, the economy bounced back strongly, but 2022 and 2023 saw rising inflation due to supply chain issues and stimulus-driven demand. In 2024, the U.S. GDP grew at a modest pace of around 2.1%, as the Federal Reserve’s interest rate hikes aimed at controlling inflation also moderated economic 

In [15]:
print("\n================ FINAL ANSWER ================")

print(answer)


================ FINAL ANSWER ================
As of 2024, the United States’ nominal GDP is estimated to be around $28 trillion USD.


# FAISS Index Mechanisms

| Index           | Search method             | Accuracy  | Speed                    | Best for              |
| --------------- | ------------------------- | --------- | ------------------------ | --------------------- |
| `IndexFlatL2`   | Exact brute-force         | Very high | Slower for huge datasets | Small/medium datasets |
| `IndexFlatIP`   | Exact dot-product         | Very high | Slower for huge datasets | Similarity search     |
| `IndexIVFFlat`  | Approximate/inverted-file | High      | Faster                   | Large datasets        |
| `IndexHNSWFlat` | Graph-based               | High      | Very fast                | Large-scale search    |
